In [1]:
import sys
from pathlib import Path
import fitz
import pandas as pd

project_root = Path.cwd().parent
sys.path.append(str(project_root))
print(sys.executable)

c:\Healthcare_Prior_Authorization_AI_Assistant\h1venv\Scripts\python.exe


In [2]:
PDF_FOLDER = Path("../data/raw")
pdf_files = list(PDF_FOLDER.glob("*.pdf"))
len(pdf_files), pdf_files

(5,
 [WindowsPath('../data/raw/Knee Arthroscopy _ Osteoarthritis of the Knee - Medical Clinical Policy Bulletins _ Aetna.pdf'),
  WindowsPath('../data/raw/Magnetic Resonance Cholangiopancreatography - Medical Clinical Policy Bulletins _ Aetna.pdf'),
  WindowsPath('../data/raw/Magnetic Resonance Imaging (MRI) and Computed Tomography (CT) of the Spine - Medical Clinical Policy Bulletins _ Aetna.pdf'),
  WindowsPath('../data/raw/Magnetic Resonance Imaging (MRI) of the Extremities - Medical Clinical Policy Bulletins _ Aetna.pdf'),
  WindowsPath('../data/raw/Magnetic Resonance Imaging of the Cardiovascular System - Cardiac MRI - Medical Clinical Policy Bulletins _ Aetna.pdf')])

In [3]:
document_info = []
for pdf in pdf_files:
    doc = fitz.open(pdf)
    text_length = sum(len(page.get_text()) for page in doc)
    document_info.append({
        "File": pdf.name,
        "Pages": len(doc),
        "Extracted Characters": text_length
    })
    doc.close()

df_docs = pd.DataFrame(document_info)
df_docs

,File,Pages,Extracted Characters
0,Knee Arthroscopy _ Osteoarthritis of the Knee ...,163,417016
1,Magnetic Resonance Cholangiopancreatography - ...,44,100382
2,Magnetic Resonance Imaging (MRI) and Computed ...,60,141515
3,Magnetic Resonance Imaging (MRI) of the Extrem...,42,95196
4,Magnetic Resonance Imaging of the Cardiovascul...,77,187008


In [4]:
spine_pdf = [
    pdf for pdf in pdf_files 
    if "Spine" in pdf.name
][0]
spine_pdf

WindowsPath('../data/raw/Magnetic Resonance Imaging (MRI) and Computed Tomography (CT) of the Spine - Medical Clinical Policy Bulletins _ Aetna.pdf')

In [5]:
from langchain_core.documents import Document

documents = []
doc = fitz.open(spine_pdf)

for page_num, page in enumerate(doc):
    text = page.get_text()
    if len(text.strip()) > 100:
        documents.append(
            Document(
                page_content=text,
                metadata={
                    "payer": "Aetna",
                    "policy_number": "0236",
                    "procedure": "MRI and CT Spine",
                    "source": spine_pdf.name,
                    "page": page_num + 1
                }
            )
        )

doc.close()
len(documents)

60

In [6]:
import importlib
import src.ingestion.clean_documents as cd
importlib.reload(cd)

cleaned_documents = cd.clean_documents(documents)
len(cleaned_documents)

60

In [7]:
print(cleaned_documents[1].page_content[:1000])
print(cleaned_documents[1].metadata)

I. Medical Necessity Aetna considers magnetic resonance imaging (MRI) and computed tomography (CT) of the spine medically necessary when any of the following criteria is met: Clinical evidence of spinal stenosis; or Clinical suspicion of a spinal cord or cauda equina compression syndrome; or Congenital anomalies or deformities of the spine; or Diagnosis and evaluation of lumbar epidural lipomatosis; or Evaluation of recurrent symptoms after spinal surgery; or Evaluation prior to epidural injection to rule out tumor or infection and to delineate the optimal anatomical location for performing the injection; or Follow-up of evaluation for spinal malignancy or spinal infection; or Known or suspected myelopathy (e.g., multiple sclerosis) for initial diagnosis when MRI of the brain is negative or symptoms mimic those of other spinal or brainstem lesions; or Known or suspected primary spinal cord tumors (malignant or non-malignant); or Persistent back or neck pain with radiculopathy as eviden

In [8]:
def filter_documents(documents):
    useful_docs = []
    remove_keywords = [
        "Table Of Contents",
        "Top",
        "Additional Information",
        "Policy History"
    ]
    for doc in documents:
        if not any(
            keyword.lower() in doc.page_content.lower()
            for keyword in remove_keywords
        ):
            useful_docs.append(doc)
    return useful_docs

filtered_documents = filter_documents(cleaned_documents)
len(filtered_documents)

52

In [9]:
from src.ingestion.chunk_documents import chunk_documents
chunks = chunk_documents(filtered_documents)
len(chunks)

c:\Healthcare_Prior_Authorization_AI_Assistant\h1venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


198

In [10]:
print(chunks[0].page_content[:1000])
print(chunks[0].metadata)

I
{'payer': 'Aetna', 'policy_number': '0236', 'procedure': 'MRI and CT Spine', 'source': 'Magnetic Resonance Imaging (MRI) and Computed Tomography (CT) of the Spine - Medical Clinical Policy Bulletins _ Aetna.pdf', 'page': 2}


In [11]:
from src.retrieval.create_vectorstore import create_vectorstore
vectorstore = create_vectorstore(chunks)
type(vectorstore)

c:\Healthcare_Prior_Authorization_AI_Assistant\src\retrieval\create_vectorstore.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2113.11it/s]


langchain_community.vectorstores.faiss.FAISS

In [12]:
query = "What are the medical necessity criteria for MRI of the spine?"
results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print("="*80)
    print(f"RESULT {i+1}")
    print(doc.page_content[:1000])
    print("\nMetadata:")
    print(doc.metadata)

RESULT 1
. Medical Necessity Aetna considers magnetic resonance imaging (MRI) and computed tomography (CT) of the spine medically necessary when any of the following criteria is met: Clinical evidence of spinal stenosis; or Clinical suspicion of a spinal cord or cauda equina compression syndrome; or Congenital anomalies or deformities of the spine; or Diagnosis and evaluation of lumbar epidural lipomatosis; or Evaluation of recurrent symptoms after spinal surgery; or Evaluation prior to epidural injection to rule out tumor or infection and to delineate the optimal anatomical location for performing the injection; or Follow-up of evaluation for spinal malignancy or spinal infection; or Known or suspected myelopathy (e.g., multiple sclerosis) for initial diagnosis when MRI of the brain is negative or

Metadata:
{'payer': 'Aetna', 'policy_number': '0236', 'procedure': 'MRI and CT Spine', 'source': 'Magnetic Resonance Imaging (MRI) and Computed Tomography (CT) of the Spine - Medical Clinic

In [13]:
import os
from dotenv import load_dotenv
load_dotenv()
print("Key loaded:", bool(os.getenv("GOOGLE_API_KEY")))

Key loaded: True


In [14]:
from src.rag.retriever import get_retriever

retriever = get_retriever(vectorstore, k=3)

query = "What are the medical necessity criteria for MRI of the spine?"
docs = retriever.invoke(query)

for i, doc in enumerate(docs):
    print("="*80)
    print(f"RESULT {i+1}")
    print(doc.page_content[:500])
    print("\nMetadata:", doc.metadata)

RESULT 1
. Medical Necessity Aetna considers magnetic resonance imaging (MRI) and computed tomography (CT) of the spine medically necessary when any of the following criteria is met: Clinical evidence of spinal stenosis; or Clinical suspicion of a spinal cord or cauda equina compression syndrome; or Congenital anomalies or deformities of the spine; or Diagnosis and evaluation of lumbar epidural lipomatosis; or Evaluation of recurrent symptoms after spinal surgery; or Evaluation prior to epidural injectio

Metadata: {'payer': 'Aetna', 'policy_number': '0236', 'procedure': 'MRI and CT Spine', 'source': 'Magnetic Resonance Imaging (MRI) and Computed Tomography (CT) of the Spine - Medical Clinical Policy Bulletins _ Aetna.pdf', 'page': 2}
RESULT 2
. According to accepted guidelines, MRI is the preferred method of imaging for each of the medically necessary indications listed in the Policy section, with the exception of (i) suspected spinal fracture or dislocation due to trauma, where CT sc

In [15]:
from src.rag.prompt import prompt

# Format retrieved docs into context string
def format_docs(docs):
    formatted = []
    for doc in docs:
        text = doc.page_content
        meta = doc.metadata
        citation = f"[Payer: {meta['payer']} | Policy: {meta['policy_number']} | Procedure: {meta['procedure']} | Page: {meta['page']}]"
        formatted.append(f"{text}\n{citation}")
    return "\n\n---\n\n".join(formatted)

# Test prompt formatting
context = format_docs(docs)
final_prompt = prompt.format(context=context, question=query)
print(final_prompt[:2000])

Human: You are a healthcare prior authorization assistant. Your job is to help hospital staff find insurance policy requirements from payer clinical policy documents.

STRICT RULES:
1. Answer ONLY using the context provided below.
2. If the context does not contain the answer, say: "The provided policy documents do not contain this information."
3. Do NOT use outside knowledge.
4. Do NOT provide medical advice or clinical decisions.
5. Always include citations in this exact format at the end:
   Source: [Payer], Policy [Policy Number] ([Procedure]), Page [Page Number]
6. If information comes from multiple pages, list all citations.

CONTEXT:
. Medical Necessity Aetna considers magnetic resonance imaging (MRI) and computed tomography (CT) of the spine medically necessary when any of the following criteria is met: Clinical evidence of spinal stenosis; or Clinical suspicion of a spinal cord or cauda equina compression syndrome; or Congenital anomalies or deformities of the spine; or Diagn

In [16]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

from src.rag.rag_chain import build_rag_chain

rag_chain = build_rag_chain(vectorstore)

query = "What are the medical necessity criteria for MRI of the spine?"
answer = rag_chain.invoke(query)

print("QUESTION:", query)
print("\n" + "="*80)
print("ANSWER:")
print(answer)

c:\Healthcare_Prior_Authorization_AI_Assistant\h1venv\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


QUESTION: What are the medical necessity criteria for MRI of the spine?

ANSWER:
Aetna considers magnetic resonance imaging (MRI) of the spine medically necessary when any of the following criteria are met:
* Clinical evidence of spinal stenosis
* Clinical suspicion of a spinal cord or cauda equina compression syndrome
* Congenital anomalies or deformities of the spine
* Diagnosis and evaluation of lumbar epidural lipomatosis
* Evaluation of recurrent symptoms after spinal surgery
* Evaluation prior to epidural injection to rule out tumor or infection and to delineate the optimal anatomical location for performing the injection
* Follow-up of evaluation for spinal malignancy or spinal infection
* Known or suspected myelopathy (e.g., multiple sclerosis) for initial diagnosis when MRI of the brain is negative

Source: Aetna, Policy 0236 (MRI and CT Spine), Page 2


In [17]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

from google import genai

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

print("Available models for your key:")
print("="*80)
for m in client.models.list():
    if "generateContent" in m.supported_actions:
        print(m.name)

Available models for your key:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-p

In [18]:
test_queries = [
    "When is MRI considered experimental or not medically necessary for the spine?",
    "Does Aetna require precertification for spine MRI?",
    "What are the criteria for MRI in cases of trauma?",
    "Is dynamic-kinetic MRI covered by Aetna?",
    "What is the coverage for pediatric spine MRI?"
]

for q in test_queries:
    print("="*80)
    print("Q:", q)
    print("-"*80)
    try:
        answer = rag_chain.invoke(q)
        print(answer)
    except Exception as e:
        print(f"ERROR: {e}")
    print()

Q: When is MRI considered experimental or not medically necessary for the spine?
--------------------------------------------------------------------------------


c:\Healthcare_Prior_Authorization_AI_Assistant\h1venv\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the provided documents, Aetna considers MRI and CT of the spine experimental, investigational, or unproven for all other indications because their clinical value for indications other than those listed in the policy has not been established. Additionally, Aetna considers the use of MRI for the further evaluation of unstable injury in neurologically intact individuals with blunt trauma after a negative cervical spine CT result to be not medically necessary. Furthermore, Aetna considers dynamic-kinetic MRI experimental, investigational, or unproven for the evaluation of the cervical spine because its effectiveness has not been established.

Source: Aetna, Policy 0236 (MRI and CT Spine), Page 3

Q: Does Aetna require precertification for spine MRI?
--------------------------------------------------------------------------------


c:\Healthcare_Prior_Authorization_AI_Assistant\h1venv\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


The provided policy documents do not contain this information.

Q: What are the criteria for MRI in cases of trauma?
--------------------------------------------------------------------------------


...

The provided policy documents do not contain this information.

Source: Aetna, Policy 0236 (MRI and CT Spine), Page 11, 14, 39, 42

Q: Is dynamic-kinetic MRI covered by Aetna?
--------------------------------------------------------------------------------


c:\Healthcare_Prior_Authorization_AI_Assistant\h1venv\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the provided policy documents, Aetna considers dynamic-kinetic MRI experimental, investigational, or unproven for the evaluation of the cervical spine because its effectiveness has not been established.

Source: Aetna, Policy 0236 (MRI and CT Spine), Page 3

Q: What is the coverage for pediatric spine MRI?
--------------------------------------------------------------------------------


c:\Healthcare_Prior_Authorization_AI_Assistant\h1venv\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


The provided policy documents do not contain this information.



In [19]:
from src.ingestion.load_documents import load_all_pdfs

all_documents = load_all_pdfs("../data/raw")

Loaded 163 pages from Knee Arthroscopy _ Osteoarthritis of the Knee - Medical Clinical Policy Bulletins _ Aetna.pdf
Loaded 44 pages from Magnetic Resonance Cholangiopancreatography - Medical Clinical Policy Bulletins _ Aetna.pdf
Loaded 60 pages from Magnetic Resonance Imaging (MRI) and Computed Tomography (CT) of the Spine - Medical Clinical Policy Bulletins _ Aetna.pdf
Loaded 42 pages from Magnetic Resonance Imaging (MRI) of the Extremities - Medical Clinical Policy Bulletins _ Aetna.pdf
Loaded 77 pages from Magnetic Resonance Imaging of the Cardiovascular System - Cardiac MRI - Medical Clinical Policy Bulletins _ Aetna.pdf

Total documents: 386


In [20]:
# Group documents by policy
from collections import defaultdict

by_policy = defaultdict(list)
for doc in all_documents:
    by_policy[doc.metadata['policy_number']].append(doc)

print(f"Policies found: {len(by_policy)}")
print()
for policy_num, docs in by_policy.items():
    print(f"Policy {policy_num}: {docs[0].metadata['procedure']} - {len(docs)} pages")

Policies found: 5

Policy 0673: Knee Arthroscopy / Osteoarthritis - 163 pages
Policy 0384: MRCP - 44 pages
Policy 0236: MRI and CT Spine - 60 pages
Policy 0171: MRI Extremities - 42 pages
Policy 0520: Cardiac MRI - 77 pages


In [21]:
import importlib
import src.ingestion.clean_documents as cd
importlib.reload(cd)

cleaned_all = cd.clean_documents(all_documents)
print(f"After cleaning: {len(cleaned_all)}")

filtered_all = filter_documents(cleaned_all)
print(f"After filtering: {len(filtered_all)}")

After cleaning: 385
After filtering: 346


In [22]:
from src.ingestion.chunk_documents import chunk_documents
chunks_all = chunk_documents(filtered_all)
print(f"Total chunks: {len(chunks_all)}")

Total chunks: 1351


In [23]:
from src.retrieval.create_vectorstore import create_vectorstore
vectorstore_all = create_vectorstore(chunks_all)
print("Vectorstore built.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3545.29it/s]


Vectorstore built.


In [24]:
rag_chain_all = build_rag_chain(vectorstore_all)

cross_policy_queries = [
    "What are the criteria for MRI of the spine?",
    "When is knee arthroscopy medically necessary?",
    "What are the coverage criteria for cardiac MRI?",
    "When is MRCP appropriate?",
    "What are the criteria for MRI of the extremities?"
]

for q in cross_policy_queries:
    print("="*80)
    print("Q:", q)
    print("-"*80)
    try:
        answer = rag_chain_all.invoke(q)
        print(answer)
    except Exception as e:
        print(f"ERROR: {e}")
    print()

Q: What are the criteria for MRI of the spine?
--------------------------------------------------------------------------------


c:\Healthcare_Prior_Authorization_AI_Assistant\h1venv\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Aetna considers magnetic resonance imaging (MRI) of the spine medically necessary when any of the following criteria are met:
- Clinical evidence of spinal stenosis
- Clinical suspicion of a spinal cord or cauda equina compression syndrome
- Congenital anomalies or deformities of the spine
- Diagnosis and evaluation of lumbar epidural lipomatosis
- Evaluation of recurrent symptoms after spinal surgery
- Evaluation prior to epidural injection to rule out tumor or infection and to delineate the optimal anatomical location for performing the injection
- Follow-up of evaluation for spinal malignancy or spinal infection
- Known or suspected myelopathy (e.g., multiple sclerosis) for initial diagnosis when MRI of the brain is negative

Source: Aetna, Policy 0236 (MRI and CT Spine), Page 2

Q: When is knee arthroscopy medically necessary?
--------------------------------------------------------------------------------


c:\Healthcare_Prior_Authorization_AI_Assistant\h1venv\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the provided policy documents, Aetna considers arthroscopic knee surgery (with or without partial meniscectomy or meniscal repair) medically necessary for persons presenting with:
- Significant knee pain plus mechanical symptoms;
- No more than mild osteoarthritis (Kellgren-Lawrence 0, 1, or 2, or modified Outerbridge Grade 0, 1, or 2);
- Radiologic confirmation of the pathology (X-ray for loose bodies, MRI for meniscal tears and/or loose bodies); and
- Failure of conservative therapy.

Additionally, knee arthroscopy is considered medically necessary and covered for persons with radiologically proven pathology that meets the specified criteria. 

Source: Aetna, Policy 0673 (Knee Arthroscopy / Osteoarthritis), Page 2
Source: Aetna, Policy 0673 (Knee Arthroscopy / Osteoarthritis), Page 3

Q: What are the coverage criteria for cardiac MRI?
--------------------------------------------------------------------------------


...

The provided policy documents do not contain this information.

Source: Aetna, Policy 0520 (Cardiac MRI), Page 15, 16, 50, 69, 73

Q: When is MRCP appropriate?
--------------------------------------------------------------------------------


c:\Healthcare_Prior_Authorization_AI_Assistant\h1venv\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the provided policy documents, MRCP is appropriate for the following:

* As a sufficient modality for the diagnosis of primary sclerosing cholangitis (PSC) in many cases of suspected PSC, allowing the risks associated with ERCP to be avoided [Source: Aetna, Policy 0384 (MRCP), Page: 14].
* As the first line choice of modality for the diagnosis of primary sclerosing cholangitis (PSC) [Source: Aetna, Policy 0384 (MRCP), Page: 14].
* For the evaluation of PSC as a replacement for ERCP [Source: Aetna, Policy 0384 (MRCP), Page: 14].
* To identify relevant biliary strictures; if a relevant stricture is observed on MRCP, it is diagnostic of PSC [Source: Aetna, Policy 0384 (MRCP), Page: 14].
* As a prognostic tool using scoring models conducted via MRI/MRCP, as such scores are associated with long-term outcomes of PSC [Source: Aetna, Policy 0384 (MRCP), Page: 14].
* To monitor small-duct PSC for the development of large-duct disease [Source: Aetna, Policy 0384 (MRCP), Page: 14].
* For

c:\Healthcare_Prior_Authorization_AI_Assistant\h1venv\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the provided policy documents, MRI of the extremities is considered appropriate for the evaluation of masses, localized infections, non-healing fractures of long bones, and in certain cases, preoperative planning. Additionally, it is listed for the following indications:
* Assessment of perfusion in diabetic foot ulcer
* Diagnosis or monitoring arthritis
* Diagnosis or prognosis of spinal cord injury and whiplash associated disorder
* Diagnosis of chronic exertional compartment syndrome
* Diagnosis suspected upper extremity deep vein thrombosis
* Evaluation and/or monitoring of disease progression in facioscapulohumeral muscular dystrophy

Source: Aetna, Policy 0171 (MRI Extremities), Page 3
Source: Aetna, Policy 0171 (MRI Extremities), Page 9



In [25]:
query = "What are the coverage criteria for cardiac MRI?"
retrieved = vectorstore_all.similarity_search(query, k=5)

for i, doc in enumerate(retrieved):
    print("="*80)
    print(f"RESULT {i+1}")
    print("Policy:", doc.metadata['policy_number'], "-", doc.metadata['procedure'])
    print("Page:", doc.metadata['page'])
    print(doc.page_content[:300])
    print()

RESULT 1
Policy: 0520 - Cardiac MRI
Page: 69
. ACCF/ACR/SCCT/SCMR/ASNC/NASCI/SCAI/SIR 2006 appropriateness criteria for cardiac computed tomography and cardiac magnetic resonance imaging: a report of the American College of Cardiology Foundation Quality Strategic Directions Committee Appropriateness Criteria Working Group, American College of 

RESULT 2
Policy: 0520 - Cardiac MRI
Page: 50
fifth, functional markers or ventricular dimensions were considered prognostically relevant in a few studies. Relevant challenges in this systematic review were the lack of comparators and reference standards relative to cardiac MRI as the index test as well as patient selection bias. The authors co

RESULT 3
Policy: 0520 - Cardiac MRI
Page: 16
(Hoffman et al, 2011) rendered a "2' rating for MRI of heart with or without stress without contrast; and a "3" rating for MRI of heart with or without stress without and with contrast for evaluation of patients with acute non- specific chest pain (rating Scale

In [26]:
# Look at all Cardiac MRI pages 1-5 in the vectorstore
cardiac_early_chunks = [
    doc for doc in chunks_all 
    if doc.metadata['policy_number'] == '0520' 
    and doc.metadata['page'] <= 5
]

print(f"Found {len(cardiac_early_chunks)} early-page chunks for Cardiac MRI")
print()
for doc in cardiac_early_chunks[:5]:
    print(f"Page {doc.metadata['page']}:")
    print(doc.page_content[:400])
    print("-"*80)

Found 7 early-page chunks for Cardiac MRI

Page 2:
American College of Cardiology Foundation, American College of Radiology (ACR) and the American Heart Association (AHA): A. Thoracic aortic disease For example: abnormal aortic contour or size on chest X-ray, differentiation of mediastinal mass versus vascular abnormality, to rule out aortic dissection, aneurysm, leaking thoracic aneurysm, exclude aortic source of peripheral embolization, Sinus Va
--------------------------------------------------------------------------------
Page 2:
. Pericardial disease For example: to assess pericardial thickness and detection of metastases, for diagnosing pericardial cysts, pericarditis and constriction, for diagnosing effusion and tamponade; or C. External or internal masses, pathology of lung and pleura For example: chest wall and mediastinal tumor invasion of the lung and pleura, masses (e.g., lipoma), intracavity tumors, and differenti
----------------------------------------------------------

In [27]:
better_queries = [
    "When is cardiac MRI medically necessary?",
    "What are the indications for cardiac MRI?",
    "What conditions does Aetna cover for cardiac MRI?"
]

for q in better_queries:
    print("="*80)
    print("Q:", q)
    print("-"*80)
    try:
        answer = rag_chain_all.invoke(q)
        print(answer)
    except Exception as e:
        print(f"ERROR: {e}")
    print()

Q: When is cardiac MRI medically necessary?
--------------------------------------------------------------------------------


c:\Healthcare_Prior_Authorization_AI_Assistant\h1venv\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


The provided policy documents do not contain this information.

Source: Aetna, Policy 0520 (Cardiac MRI), Page 13, 14, 15, 68, 74

Q: What are the indications for cardiac MRI?
--------------------------------------------------------------------------------


c:\Healthcare_Prior_Authorization_AI_Assistant\h1venv\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Under established guidelines, MRI is used as the diagnostic test for the following indications:
* Diseases of the aorta
* Diseases of the pericardium
* External and internal masses
* Pathology involving surrounding structures
* Congenital heart disease
* Ventricular dysplasia

Additionally, cardiac MRI may be warranted when echocardiography does not provide enough information, and it is used for the management of cardiotoxicity caused by cancer therapies, including the evaluation of cardiac masses and cardiac function, and to differentiate benign and malignant primary cardiac tumors, metastatic disease, and pseudo-tumors.

Source: Aetna, Policy 0520 (Cardiac MRI), Page 14
Source: Aetna, Policy 0520 (Cardiac MRI), Page 31

Q: What conditions does Aetna cover for cardiac MRI?
--------------------------------------------------------------------------------


c:\Healthcare_Prior_Authorization_AI_Assistant\h1venv\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


The provided policy documents do not contain this information.

Source: Aetna, Policy 0520 (Cardiac MRI), Page 43
Source: Aetna, Policy 0520 (Cardiac MRI), Page 77



In [28]:
from src.retrieval.create_vectorstore import save_vectorstore

save_vectorstore(vectorstore_all, "../data/processed/faiss_index")

Vectorstore saved to: ..\data\processed\faiss_index


In [29]:
from src.retrieval.create_vectorstore import load_vectorstore

loaded_vs = load_vectorstore("../data/processed/faiss_index")

# Quick test
results = loaded_vs.similarity_search("MRI spine medical necessity", k=2)
print(f"Loaded {loaded_vs.index.ntotal} vectors")
print(f"First result page: {results[0].metadata['page']}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8289.47it/s]


Vectorstore loaded from: ..\data\processed\faiss_index
Loaded 1351 vectors
First result page: 2
